# LO6 - LangGraph 기반 Agent 구현

앞 슬롯에서 만든 도구를 LangGraph Agent에 연결합니다. 먼저 `ToolNode`와 `tools_condition`으로 수동 그래프를 짜서 내부 동작을 확인한 뒤, `create_agent` 한 줄로 같은 Agent를 재현해 둘을 대조합니다. 메모리(대화 누적)는 다음 슬롯의 주제이므로 여기서는 단발 실행만 다룹니다.

## 학습 목표

- `ToolNode`와 `tools_condition`으로 도구 호출 분기를 갖춘 수동 Agent 그래프를 구성한다.
- ReAct 실행 루프(추론-행동-관찰)가 그래프의 순환으로 표현되는 것을 확인한다.
- 수동 그래프와 `create_agent` 한 줄 버전을 비교하고, `create_react_agent`(deprecated)와 대조한다.

> 실제 실행에는 API 키가 필요합니다. 강의 직전 모델명과 가격을 재확인하십시오.

## 0단계: 패키지 설치(핀 고정)

강의 시점 검증 버전으로 고정합니다. 설치 후 런타임 재시작을 권장합니다.

In [ ]:
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai

## API 키 설정

Colab에서는 좌측 열쇠 아이콘(`userdata`)에 `OPENAI_API_KEY`를 저장한 뒤 아래 셀로 불러옵니다. 로컬에서는 환경변수나 `.env`로 대신할 수 있습니다.

In [ ]:
import os

# Colab: 좌측 열쇠 아이콘에 OPENAI_API_KEY를 저장한 뒤 불러옵니다
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 로컬 대안: 환경변수에 직접 설정하거나 .env를 사용합니다
# os.environ["OPENAI_API_KEY"] = "sk-..."
# from dotenv import load_dotenv; load_dotenv()

# 대안 모델(google-genai) 사용 시: os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## 1단계: 모델과 도구 준비

LO4에서 만든 도구를 그래프에 연결할 수 있도록 모델에 바인딩합니다. `@tool`의 docstring이 곧 도구 설명이며, 모델은 이 설명을 보고 어느 도구를 부를지 라우팅합니다.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# 강의 직전 최신 모델과 가격을 재확인하십시오.
model = init_chat_model("openai:gpt-5.4-mini")
# 대안: init_chat_model("google-genai:gemini-2.5-flash")


@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""  # docstring이 도구 설명이며 라우팅 기준이 됩니다
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱한다."""
    return a * b


tools = [add, multiply]                       # LO4에서 만든 도구를 그래프에 연결
model_with_tools = model.bind_tools(tools)    # 모델에 도구 스키마 바인딩

## 2단계: 수동 Agent 그래프(ToolNode + tools_condition)

모델 노드에서 출발해 `tools_condition`으로 분기하고, 도구 노드는 실행 후 다시 모델 노드로 돌아갑니다. 이 되돌아오는 엣지가 Agent의 순환을 만듭니다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition  # 사전 제작 부품


class State(TypedDict):
    messages: Annotated[list, add_messages]   # reducer: 메시지 누적(append)


def call_model(state: State):
    # 모델이 답하거나 도구 호출을 요청합니다 (추론과 행동 결정)
    # model_with_tools는 1단계 셀의 전역 변수를 참조하므로 셀을 순서대로 실행하십시오
    return {"messages": [model_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))        # 도구 실행 노드
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)  # 도구 호출 있으면 tools, 없으면 END
builder.add_edge("tools", "model")                # 관찰 결과를 들고 다시 추론(순환)
agent = builder.compile()

result = agent.invoke({"messages": [{"role": "user", "content": "3 더하기 5를 4와 곱하면?"}]})
print(result["messages"][-1].content)             # 최종 답변
# 예상 출력: "3 더하기 5는 8이고, 8과 4를 곱하면 32입니다." (add 한 번, multiply 한 번 호출)

# 변형 포인트: builder.add_edge("tools", "model") 줄을 지우면
# 도구 실행 후 모델로 돌아가지 못해 순환이 끊기고, 도구 결과를 정리하는 단계가 사라집니다.

**체크포인트**: 수동 그래프가 도구를 호출하고 그 결과를 반영해 최종 답변을 내는지 확인했습니다. 도구 노드에서 모델 노드로 돌아가는 엣지가 ReAct 루프의 순환을 만듭니다.

## 3단계: 실행 흐름 관찰(ReAct 루프 확인)

`stream`으로 노드가 모델, 도구, 다시 모델 순으로 오가는 것을 단계별로 확인합니다.

In [ ]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "3 더하기 5를 4와 곱하면?"}]},
    stream_mode="values",
):
    m = step["messages"][-1]  # 각 단계의 마지막 메시지(추론-행동-관찰)
    # 메시지 객체면 pretty_print, dict로 오면 그대로 출력 (버전별 형태 방어)
    m.pretty_print() if hasattr(m, "pretty_print") else print(m)

# 예상 출력 흐름(메시지 종류로 단계가 드러남):
#   HumanMessage  : 사용자 질문
#   AIMessage     : tool_calls=[add(3,5)]  (추론 후 도구 호출 결정)
#   ToolMessage   : "8"                    (도구 실행 결과 관찰)
#   AIMessage     : tool_calls=[multiply(8,4)]
#   ToolMessage   : "32"
#   AIMessage     : 최종 답변 (도구 호출 없음이므로 tools_condition이 END로 보냄)
# 변형 포인트: 질문을 "3 더하기 5는?"으로 바꾸면 multiply 단계 없이 add 한 번으로 끝납니다.
# 모델이 필요한 도구만 골라 부른다는 것(멀티툴 라우팅)을 직접 볼 수 있습니다.

**체크포인트**: `tools_condition`이 도구 호출 유무로 분기하고, 도구 호출이 없는 마지막 메시지에서 루프가 종료되는 것을 stream 출력에서 관찰했습니다.

## 4단계: create_agent 한 줄 버전으로 대조

`langchain.agents`의 `create_agent`로 수동 그래프와 동일한 ReAct Agent를 한 줄에 만듭니다. v1.0 권장 경로입니다.

In [ ]:
from langchain.agents import create_agent      # v1.0 권장 경로

# 위 수동 그래프와 동일한 ReAct Agent를 한 줄로 생성
agent2 = create_agent(
    "openai:gpt-5.4-mini",                      # 강의 직전 최신 모델과 가격을 재확인하십시오
    tools=tools,                                # 같은 도구 목록 재사용
    system_prompt="너는 정확한 계산 비서다.",   # 시스템 프롬프트(LO4 설계 적용)
)
out = agent2.invoke({"messages": [{"role": "user", "content": "3 더하기 5를 4와 곱하면?"}]})
print(out["messages"][-1].content)              # 수동 그래프와 같은 결과 확인
# 예상 출력: 2단계 수동 그래프와 동일한 답("...32입니다.")이 나옵니다.

In [ ]:
# 변형 포인트: recursion_limit으로 루프 폭주를 막는 안전망을 직접 확인
# 한도를 비현실적으로 낮추면(예: 2) 도구를 다 돌기 전에 막혀 에러가 납니다.
from langgraph.errors import GraphRecursionError

try:
    agent2.invoke(
        {"messages": [{"role": "user", "content": "3 더하기 5를 4와 곱하면?"}]},
        {"recursion_limit": 2},                 # 한 실행의 최대 스텝 수. 무한 루프 차단용 상한
    )
except GraphRecursionError as e:
    print("스텝 상한 초과로 중단:", e)           # 한도 초과 시 GraphRecursionError 발생

# 참고: 인터넷 예제의 구버전(현재 동작하나 deprecated, 2.0 제거 예정)
# from langgraph.prebuilt import create_react_agent
# agent_old = create_react_agent(model, tools, prompt="너는 정확한 계산 비서다.")

**체크포인트**: `create_agent` 한 줄 버전이 수동 그래프와 같은 답변을 내고, `recursion_limit`을 낮게 주면 `GraphRecursionError`로 루프가 차단되는 것을 확인했습니다. `create_react_agent`는 deprecated 대조용임을 구분했습니다.

## 5단계: LO4 재고 도구를 얹어 "재고 Agent" 만들기

앞 단계까지는 add/multiply 같은 계산 도구로 구조를 익혔습니다. 이번에는 LO4에서 만든 업무용 도구(`check_inventory`)와 시스템 프롬프트를 그대로 `create_agent`에 얹어, 슬롯3의 산출물이 어떻게 Agent로 이어지는지 확인합니다. 도구와 프롬프트만 바꿔 끼우면 같은 한 줄로 업무 Agent가 완성됩니다.

In [ ]:
# LO4(Custom Tool 개발)에서 만든 도구와 시스템 프롬프트를 그대로 재사용
from pydantic import BaseModel, Field
from langchain_core.tools import tool, ToolException


class InventoryInput(BaseModel):
    sku: str = Field(description="조회할 제품 코드. 예: 'BAT-21700'")
    warehouse: str = Field("ICN", description="창고 코드. 예: 'ICN', 'PUS'")


_STOCK = {("BAT-21700", "ICN"): 1240, ("BAT-21700", "PUS"): 380}


@tool("check_inventory", args_schema=InventoryInput)
def check_inventory(sku: str, warehouse: str = "ICN") -> str:
    """지정한 창고의 제품 재고 수량을 조회한다."""
    qty = _STOCK.get((sku, warehouse))
    if qty is None:
        raise ToolException(f"재고 정보 없음: sku={sku}, warehouse={warehouse}")
    return f"{warehouse} 창고의 {sku} 재고는 {qty}개입니다."


# LO4의 시스템 프롬프트(역할, 도구 사용 규칙, 출력 형식)를 그대로 적용
inventory_agent = create_agent(
    "openai:gpt-5.4-mini",                       # 강의 직전 최신 모델과 가격을 재확인하십시오
    tools=[check_inventory],                      # LO4에서 만든 업무용 도구
    system_prompt=(
        "너는 사내 재고를 조회하는 물류 비서다. "
        "재고 수량은 추측하지 말고 반드시 check_inventory 도구로 확인하라. "
        "답변은 수량을 숫자와 단위를 함께 한 문장으로 답하라."
    ),
)
res = inventory_agent.invoke(
    {"messages": [{"role": "user", "content": "BAT-21700 인천 창고 재고 얼마야?"}]}
)
print(res["messages"][-1].content)                # 재고 도구를 호출해 답하는지 확인

**체크포인트**: LO4의 재고 도구와 시스템 프롬프트를 `create_agent`에 얹어 "재고 Agent"가 도구를 호출해 답하는 것을 확인했습니다.

## 참고 자료

| 자료 | 설명 |
|------|------|
| [LangGraph Agents - ToolNode / tools_condition](https://docs.langchain.com/oss/python/langgraph/graph-api) | 도구 노드와 조건 분기 사전 제작 부품 |
| [LangChain Agents - create_agent](https://docs.langchain.com/oss/python/langchain/agents) | v1.0 권장 Agent 생성 함수 |
| [ReAct: Synergizing Reasoning and Acting (Yao et al., 2022)](https://arxiv.org/abs/2210.03629) | ReAct 패턴 원논문 |